In [1]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

I0000 00:00:1790204152.952638    7856 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790204153.057481    7856 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790204154.763045    7856 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [9]:
with open("ocean_proximity_encoder.pkl", "rb") as file:
    encoder = pickle.load(file)

with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

model = load_model("model.keras")

In [10]:
print(type(encoder))

<class 'sklearn.preprocessing._encoders.OneHotEncoder'>


In [11]:
user_data = pd.DataFrame({
    "longitude": [-118.30],
    "latitude": [34.05],
    "housing_median_age": [25],
    "total_rooms": [3000],
    "total_bedrooms": [600],
    "population": [1200],
    "households": [500],
    "median_income": [5.25],
    "ocean_proximity": ["NEAR OCEAN"]
})

In [12]:
encoded=encoder.transform(user_data[["ocean_proximity"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["ocean_proximity"]),
    index=user_data.index
)

In [13]:
user_data = pd.concat(
    [
        user_data.drop("ocean_proximity", axis=1),
        encoded_df
    ],
    axis=1
)

In [14]:
user_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-118.3,34.05,25,3000,600,1200,500,5.25,0.0,0.0,0.0,0.0,1.0


In [15]:
user_data_scaled = scaler.transform(user_data)

/home/debmalyaa/anaconda3/envs/tensorflow-gpu/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [16]:
user_data_scaled

array([[0.60258964, 0.15957447, 0.47058824, 0.07625006, 0.09295469,
        0.03354915, 0.08205887, 0.32758858, 0.        , 0.        ,
        0.        , 0.        , 1.        ]])

In [17]:
prediction = model.predict(user_data_scaled)

print("Predicted House Price:", prediction[0][0])

I0000 00:00:1790205554.194871    8297 service.cc:153] XLA service 0x7fe37c042520 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790205554.194901    8297 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.26.0)
I0000 00:00:1790205554.224674    8297 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790205554.263939    8297 cuda_dnn.cc:461] Loaded cuDNN version 92600


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 641ms/step
Predicted House Price: 261557.05


I0000 00:00:1790205554.747990    8297 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
